# Cross-Validation

Use K-fold cross-validation to estimate model performance more reliably than a single split.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
data = load_breast_cancer(as_frame=True); X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print('Fold scores:', np.round(scores, 4)); print('Mean:', round(scores.mean(), 4)); print('Standard deviation:', round(scores.std(), 4))

Fold scores: [0.967 0.989 0.978 0.989 0.967]
Mean: 0.978
Standard deviation: 0.0098


## Compare scoring metrics
Use metrics that reflect the problem and class balance.

In [2]:
rows = []
for scoring in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc']:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({'scoring': scoring, 'mean': scores.mean(), 'std': scores.std()})
pd.DataFrame(rows).round(4)

,scoring,mean,std
0,accuracy,0.9780,0.0098
1,precision_macro,0.9782,0.0099
2,recall_macro,0.9753,0.0126
3,f1_macro,0.9764,0.0106
4,roc_auc,0.9959,0.0050


## Final held-out test evaluation
Use cross-validation on training data for model selection. Keep the test set for final evaluation.

In [3]:
model.fit(X_train, y_train)
print('Held-out test accuracy:', round(accuracy_score(y_test, model.predict(X_test)), 4))

Held-out test accuracy: 0.9825


## Key takeaways
Stratified K-fold approximately preserves class proportions. Report mean and variability. Put preprocessing in a pipeline to avoid leakage, and avoid repeatedly tuning against the test set.